# N159 · Pandas日期、排名与滚动统计

**目标：** 用矢量化运算表达时间和分组窗口。

**先修：** N158, N155, N156。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** to_datetime；dt访问；rank；groupby shift；rolling；字符串str；缺失日期。

## 从问题到算法

“前三名”先问并列算几个名次。本章用部门内薪资 `dense` 排名，第三个不同薪资的所有员工都保留，而不是只截取三行。

次日留存的分母是玩家数，不是日志行数；先对玩家—日期去重，取各自首次日期，再判断次日是否存在。自然日跨月仍是加一天，不能把日期字符串最后一位加一。

七日营业额窗口为 `(当日−7天, 当日]`。先按日聚合，再做时间窗口。缺失日视为零，但输出只保留原表出现、且已经历七个自然日的日期。按行数 `rolling(7)` 在日期缺口存在时是另一道题。金额限定为可精确表示的有界整数；这不是通用财务精度库。

## 最小实现

**本章接口：** `department_top_three_salaries(employees, departments)`、`next_day_retention(activity)`、`normalize_names(users)`、`daily_rolling_sales(df)`

In [1]:
import pandas as pd
from pandas.testing import assert_frame_equal

from decimal import Decimal, ROUND_HALF_UP

def _round2(value):
    return float(Decimal(str(value)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP))

def department_top_three_salaries(employees, departments):
    ranked = employees.copy()
    ranked['_rank'] = ranked.groupby('departmentId')['salary'].rank(method='dense', ascending=False)
    kept = ranked.loc[ranked['_rank'] <= 3].merge(
        departments.rename(columns={'id':'departmentId','name':'Department'}),
        on='departmentId',how='inner',validate='many_to_one')
    return kept.rename(columns={'name':'Employee','salary':'Salary'})[
        ['Department','Employee','Salary']].sort_values(
        ['Department','Salary','Employee'],ascending=[True,False,True]).reset_index(drop=True)

def next_day_retention(activity):
    if activity.empty:
        return 0.0
    days = activity[['player_id','event_date']].copy()
    days['event_date'] = pd.to_datetime(days['event_date']).dt.normalize()
    days = days.drop_duplicates()
    first = days.groupby('player_id',as_index=False)['event_date'].min()
    first['event_date'] += pd.Timedelta(days=1)
    hits = first.merge(days,on=['player_id','event_date'],validate='one_to_one')
    return _round2(Decimal(len(hits)) / Decimal(len(first)))

def normalize_names(users):
    """ASCII alphabetic names; retain schema and sort by user_id."""
    result = users.copy()
    result['name'] = result['name'].str.capitalize()
    return result.sort_values('user_id').reset_index(drop=True)

def daily_rolling_sales(df):
    """Seven natural days, missing days contribute zero; output observed days only."""
    data = df[['visited_on','amount']].copy()
    data['visited_on'] = pd.to_datetime(data['visited_on']).dt.normalize()
    daily = data.groupby('visited_on')['amount'].sum().sort_index()
    if daily.empty:
        return pd.DataFrame({'visited_on':pd.Series(dtype='datetime64[ns]'),
                             'amount':pd.Series(dtype='int64'),
                             'average_amount':pd.Series(dtype='float64')})
    total = daily.rolling('7D',closed='right').sum()
    total = total.loc[total.index >= daily.index.min()+pd.Timedelta(days=6)]
    # The teaching inputs are bounded integer amounts, exactly representable as float64.
    result = total.astype('int64').rename('amount').reset_index()
    result['average_amount'] = [_round2(Decimal(int(x))/Decimal(7)) for x in result['amount']]
    return result

## 正确性、前提与复杂度

先聚合和去重决定统计单位，再做排名或日期对齐；顺序不可任意交换。时间窗口查询的是日历范围，跨月与缺口不改变定义。留存通过与唯一玩家—次日键一对一匹配，保证每个玩家最多贡献一次。

**代价：** 分组通常期望线性；排序和排名通常 O(n log n)。时间滑窗在已排序序列上处理，避免每一行重新扫描全表；存储 O(n)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

sales = pd.DataFrame({'visited_on':['2024-01-01','2024-01-01','2024-01-03','2024-01-07','2024-01-08'], 'amount':[10,20,30,70,80]})
display(sales)
display(daily_rolling_sales(sales))
activity = pd.DataFrame({'player_id':[1,1,1,2,3], 'event_date':['2024-01-31','2024-02-01','2024-02-01','2024-01-01','2024-01-02']})
show_table(['quantity','value'],[['raw rows',len(activity)],['players',activity.player_id.nunique()],['next-day fraction',next_day_retention(activity)]])

,visited_on,amount
0,2024-01-01,10
1,2024-01-01,20
2,2024-01-03,30
3,2024-01-07,70
4,2024-01-08,80


/tmp/ipykernel_4946/307785585.py:46: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  total = total.loc[total.index >= daily.index.min()+pd.Timedelta(days=6)]


,visited_on,amount,average_amount
0,2024-01-07,130,18.57
1,2024-01-08,180,25.71


/tmp/ipykernel_4946/307785585.py:26: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  first['event_date'] += pd.Timedelta(days=1)


quantity,value
raw rows,5
players,3
next-day fraction,0.33


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
employees = pd.DataFrame({'id':[1,2,3,4,5,6], 'name':['a','b','c','d','e','f'], 'salary':[100,90,90,80,70,10], 'departmentId':[1,1,1,1,1,2]})
departments = pd.DataFrame({'id':[1,2], 'name':['IT','HR']})
expected = pd.DataFrame({'Department':['HR','IT','IT','IT','IT'], 'Employee':['f','a','b','c','d'], 'Salary':[10,100,90,90,80]})
assert_frame_equal(department_top_three_salaries(employees,departments),expected)
a = pd.DataFrame({'player_id':[1,1,1,2,3], 'event_date':['2024-01-31','2024-02-01','2024-02-01','2024-01-01','2024-01-02']})
assert next_day_retention(a) == 0.33
assert next_day_retention(a.iloc[:0]) == 0.0
one_eighth = pd.DataFrame({'player_id':list(range(8))+[0], 'event_date':['2024-02-28']*8+['2024-02-29']})
assert next_day_retention(one_eighth) == 0.13
users = pd.DataFrame({'user_id':[2,1], 'name':['aLICE','bOB']})
assert_frame_equal(normalize_names(users),pd.DataFrame({'user_id':[1,2], 'name':['Bob','Alice']}))
sales = pd.DataFrame({'visited_on':['2024-01-01','2024-01-01','2024-01-03','2024-01-07','2024-01-08'], 'amount':[10,20,30,70,80]})
expected = pd.DataFrame({'visited_on':pd.to_datetime(['2024-01-07','2024-01-08']), 'amount':[130,180], 'average_amount':[18.57,25.71]})
assert_frame_equal(daily_rolling_sales(sales),expected)
assert daily_rolling_sales(sales.iloc[:0]).empty
import sqlite3
query = """WITH daily AS (
 SELECT visited_on,SUM(amount) AS value FROM Sales GROUP BY visited_on
), totals AS (
 SELECT visited_on,SUM(value) OVER (ORDER BY JULIANDAY(visited_on)
 RANGE BETWEEN 6 PRECEDING AND CURRENT ROW) AS amount FROM daily
) SELECT visited_on,amount,ROUND(amount/7.0,2) AS average_amount FROM totals
 WHERE JULIANDAY(visited_on)-(SELECT JULIANDAY(MIN(visited_on)) FROM daily)>=6
 ORDER BY visited_on"""
with sqlite3.connect(':memory:') as conn:
    sales.to_sql('Sales',conn,index=False)
    actual_sql = pd.read_sql_query(query,conn,parse_dates=['visited_on'])
assert_frame_equal(daily_rolling_sales(sales),actual_sql)
print('N159: 所有本章断言通过')

N159: 所有本章断言通过


/tmp/ipykernel_4946/307785585.py:26: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  first['event_date'] += pd.Timedelta(days=1)
/tmp/ipykernel_4946/307785585.py:26: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  first['event_date'] += pd.Timedelta(days=1)
/tmp/ipykernel_4946/307785585.py:46: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  total = total.loc[total.index >= daily.index.min()+pd.Timedelta(days=6)]
/tmp/ipykernel_4946/307785585.py:46: DeprecationWarning: The 'generic' unit for 

## 练习：先独立完成

**练习 1：** 区别按行rolling和按时间窗口rolling。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 避免无必要逐行apply。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 185. Department Top Three Salaries（comparison）
- 550. Game Play Analysis IV（comparison）
- 1667. Fix Names in a Table（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。